# 🔍 Day 01a: SQL SELECT Basics — WHERE, ORDER BY, LIKE, CASE WHEN

---

## 🎯 What You'll Master Today
- SELECT, WHERE, ORDER BY, LIMIT, DISTINCT
- LIKE, IN, BETWEEN, IS NULL
- CASE WHEN expressions
- GROUP BY, HAVING, aggregate functions

---

```
╔══════════════════════════════════════════════════════════════╗
║           SQL QUERY EXECUTION ORDER                          ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Written:  SELECT → FROM → WHERE → GROUP BY → HAVING →      ║
║            ORDER BY → LIMIT                                  ║
║                                                              ║
║  Executed: FROM → WHERE → GROUP BY → HAVING →                ║
║            SELECT → ORDER BY → LIMIT                         ║
║                                                              ║
║  💡 WHERE filters rows BEFORE grouping                      ║
║  💡 HAVING filters groups AFTER grouping                    ║
║  💡 Can't use aliases in WHERE (SELECT runs after WHERE)    ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# Setup: In-memory sample data using Python
# (simulates SQL without needing a database)
# ============================================

employees = [
    {"id": 1, "name": "Alice",   "dept": "Engineering", "salary": 95000, "mgr_id": None},
    {"id": 2, "name": "Bob",     "dept": "Engineering", "salary": 85000, "mgr_id": 1},
    {"id": 3, "name": "Charlie", "dept": "Marketing",   "salary": 72000, "mgr_id": 1},
    {"id": 4, "name": "Diana",   "dept": "Marketing",   "salary": 68000, "mgr_id": 3},
    {"id": 5, "name": "Eve",     "dept": "Engineering", "salary": 92000, "mgr_id": 1},
    {"id": 6, "name": "Frank",   "dept": "Sales",       "salary": 65000, "mgr_id": None},
    {"id": 7, "name": "Grace",   "dept": "Sales",       "salary": 71000, "mgr_id": 6},
    {"id": 8, "name": "Hank",    "dept": "Engineering", "salary": 98000, "mgr_id": 1},
]

def print_table(rows, columns=None):
    if not rows:
        print("  (empty)")
        return
    if columns is None:
        columns = list(rows[0].keys())
    widths = {c: max(len(str(c)), max(len(str(r.get(c, ''))) for r in rows)) for c in columns}
    header = "  " + " | ".join(f"{c:<{widths[c]}}" for c in columns)
    print(header)
    print("  " + "-+-".join("-" * widths[c] for c in columns))
    for r in rows:
        print("  " + " | ".join(f"{str(r.get(c, '')):<{widths[c]}}" for c in columns))

print("  Employee Table:")
print_table(employees)

In [ ]:
# ============================================
# 1. SELECT + WHERE + ORDER BY
# ============================================

# SQL: SELECT name, salary FROM employees WHERE salary > 80000 ORDER BY salary DESC;
print("  Q: SELECT name, salary WHERE salary > 80000 ORDER BY salary DESC")
result = sorted(
    [{"name": e["name"], "salary": e["salary"]} for e in employees if e["salary"] > 80000],
    key=lambda x: x["salary"], reverse=True
)
print_table(result)
print()

# SQL: SELECT DISTINCT dept FROM employees;
print("  Q: SELECT DISTINCT dept")
depts = list({e["dept"] for e in employees})
for d in sorted(depts):
    print(f"    {d}")
print()

# SQL: SELECT * FROM employees WHERE dept IN ('Engineering', 'Sales') AND salary BETWEEN 70000 AND 95000;
print("  Q: WHERE dept IN ('Engineering','Sales') AND salary BETWEEN 70000 AND 95000")
result = [e for e in employees if e["dept"] in ("Engineering", "Sales") and 70000 <= e["salary"] <= 95000]
print_table(result)
print()

# SQL: SELECT * FROM employees WHERE mgr_id IS NULL;
print("  Q: WHERE mgr_id IS NULL (top-level employees)")
result = [e for e in employees if e["mgr_id"] is None]
print_table(result)

In [ ]:
# ============================================
# 2. GROUP BY + Aggregates + HAVING
# ============================================

from collections import defaultdict

# SQL: SELECT dept, COUNT(*) as cnt, AVG(salary) as avg_sal, MAX(salary) as max_sal
#      FROM employees GROUP BY dept HAVING COUNT(*) >= 2 ORDER BY avg_sal DESC;

print("  Q: GROUP BY dept, HAVING COUNT(*) >= 2")
groups = defaultdict(list)
for e in employees:
    groups[e["dept"]].append(e["salary"])

result = []
for dept, salaries in groups.items():
    if len(salaries) >= 2:  # HAVING
        result.append({
            "dept": dept,
            "count": len(salaries),
            "avg_salary": round(sum(salaries) / len(salaries)),
            "max_salary": max(salaries),
            "min_salary": min(salaries),
        })

result.sort(key=lambda x: x["avg_salary"], reverse=True)
print_table(result)

In [ ]:
# ============================================
# 3. CASE WHEN + LIKE
# ============================================

# SQL: SELECT name, salary,
#        CASE WHEN salary >= 90000 THEN 'Senior'
#             WHEN salary >= 70000 THEN 'Mid'
#             ELSE 'Junior' END AS level
#      FROM employees ORDER BY salary DESC;

print("  Q: CASE WHEN salary tiers")
result = []
for e in employees:
    if e["salary"] >= 90000:
        level = "Senior"
    elif e["salary"] >= 70000:
        level = "Mid"
    else:
        level = "Junior"
    result.append({"name": e["name"], "salary": e["salary"], "level": level})

result.sort(key=lambda x: x["salary"], reverse=True)
print_table(result)
print()

# LIKE patterns
print("  LIKE Pattern Reference:")
patterns = [
    ("'A%'",     "Starts with A",           [e["name"] for e in employees if e["name"].startswith("A")]),
    ("'%e'",     "Ends with e",             [e["name"] for e in employees if e["name"].endswith("e")]),
    ("'_____'",  "Exactly 5 characters",    [e["name"] for e in employees if len(e["name"]) == 5]),
    ("'%an%'",   "Contains 'an'",           [e["name"] for e in employees if "an" in e["name"].lower()]),
]

for pattern, desc, matches in patterns:
    print(f"    LIKE {pattern:<12} ({desc:<22}) → {matches}")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | WHERE vs HAVING? | WHERE filters before GROUP BY. HAVING filters after (on aggregates) |
| 2 | SQL execution order? | FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT |
| 3 | COUNT(*) vs COUNT(col)? | COUNT(*): all rows. COUNT(col): non-NULL values in col |
| 4 | Can you use alias in WHERE? | No — SELECT runs after WHERE. Use subquery or CTE instead |
| 5 | NULL in comparisons? | NULL = NULL is NULL (not TRUE). Use IS NULL / IS NOT NULL |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Execution order ≠ written order (FROM first!)         │
## │  • WHERE: row filter. HAVING: group filter               │
## │  • CASE WHEN: SQL's if/else for computed columns         │
## │  • NULL is special: use IS NULL, not = NULL              │
## │  • LIKE: % = any chars, _ = single char                  │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **JOINs** — INNER, LEFT, RIGHT, FULL OUTER, CROSS, SELF